# CATE estimation

ATE tells us the average effect across everyone. CATE asks a more useful question for targeting: what's the ad's effect *conditional on* a user's features — does it vary by user, and if so, who benefits most (or is actively hurt)?

Meta-learners covered here, each motivated by a specific flaw in the previous one:
- **S-learner** (foil): one model, treatment as just another feature. Tends to shrink CATE toward zero when the treatment effect is small relative to what covariates explain.
- **T-learner**: two separate models (treated-only, control-only). Fixes S-learner's shrinkage, but its precision is bottlenecked by the smaller group (control, ~150K rows) — the T-learner subtracts one model's raw prediction from another's, so noise from the weaker model passes straight through.
- **X-learner**: designed for unbalanced treatment groups (our 85/15 split). Cross-imputes counterfactuals using the *other* group's model, then fits new models on those imputed effects — this launders the weaker model's noise through a large-sample regression instead of using it raw, and blends the two resulting estimates by propensity score.
- **DR-learner** and **causal forest**: to follow.

**Constraint to hold throughout: never evaluate a CATE model on data used to fit it.** Unlike ATE (a direct calculation, no fitting involved), CATE models are flexible ML models that can overfit — and because no individual ground-truth treatment effect ever exists to check against, an overfit model gives zero warning signal unless checked on a held-out split."

In [ ]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample, split_train_eval

df = load_sample()
train, eval_ = split_train_eval(df)

print("train:", train.shape, "eval:", eval_.shape)
print("train treatment rate:", train["treatment"].mean())
print("eval treatment rate:", eval_["treatment"].mean())

## Fit S-learner, T-learner, X-learner on `visit`

All use `HistGradientBoostingClassifier` as the base learner (faster than plain `GradientBoostingClassifier` at this row count). Fit on `train`, predict on the held-out `eval_` set only.

In [ ]:
from uplift.cate_models import (
    fit_s_learner, predict_s_learner,
    fit_t_learner, predict_t_learner,
    fit_x_learner, predict_x_learner,
)
from uplift.ate import compute_ate

outcome = "visit"

s_model = fit_s_learner(train, outcome)
s_cate = predict_s_learner(s_model, eval_)

t_models = fit_t_learner(train, outcome)
t_cate = predict_t_learner(t_models, eval_)

x_model = fit_x_learner(train, outcome)
x_cate = predict_x_learner(x_model, eval_)

In [ ]:
import pandas as pd
import numpy as np

ate_eval = compute_ate(eval_, outcome).ate

comparison = pd.DataFrame({
    "mean_predicted_cate": [s_cate.mean(), t_cate.mean(), x_cate.mean()],
    "std_predicted_cate": [s_cate.std(), t_cate.std(), x_cate.std()],
}, index=["S-learner", "T-learner", "X-learner"])

print(f"Actual ATE on eval set: {ate_eval:.5f}")
comparison

## Interpretation

- **S-learner mean is farthest from the true ATE** — the shrinkage-toward-zero effect, since a tree-based model with `treatment` as just one of 13 features rarely splits on it when covariates dominate.
- **T-learner mean sits closer to the true ATE**, but its std is the widest — the raw subtraction of two independently-fit models lets `model_control`'s noise (fit on only ~150K rows) pass straight through.
- **X-learner keeps T-learner's closer-to-true mean while pulling std back down** close to S-learner's level — cross-imputation launders the noisy `model_control` predictions through a large-sample regression fit (`tau1`, fit on ~850K treated rows) instead of using them raw, so it gets T-learner's reduced shrinkage without inheriting all of its noise.

Still open: is the remaining spread in each model's predicted CATE *real heterogeneity*, or still partly noise? Can't check per-user (no ground truth) — needs the held-out decile/Qini evaluation next.

## DR-learner (doubly robust)

Every model so far relies entirely on outcome modeling (predict the outcome from features). If the outcome model is wrong in some way, the CATE estimate inherits that error with no safety net.

The DR-learner combines outcome modeling with a second, independent approach — propensity weighting — into one pseudo-outcome per person:

```
phi_i = [mu1(x_i) - mu0(x_i)]                                   # T-learner-style estimate
        + (t_i / e(x_i)) * (y_i - mu1(x_i))                     # correction if treated
        - ((1-t_i) / (1-e(x_i))) * (y_i - mu0(x_i))              # correction if control
```

"Doubly robust" is a specific property, not just averaging: this estimator is consistent if *either* the outcome model (`mu1`/`mu0`) *or* the propensity model (`e(x)`) is correctly specified — even if the other one is wrong. If `mu1`/`mu0` are perfect, the correction terms (residuals) average to zero and `phi` reduces to the plain outcome-model estimate. If they're imperfect, the propensity-weighted residual nudges the estimate back toward what the actual observed outcomes say.

`e(x)` is a constant here (~0.85) since `treatment` was unconditionally randomized — no need to fit a separate propensity model.

Final step: fit one smoothing regression model on all of `phi` (treated and control combined) to get `tau(x)`.

In [ ]:
from uplift.cate_models import fit_dr_learner, predict_dr_learner

dr_model = fit_dr_learner(train, outcome)
dr_cate = predict_dr_learner(dr_model, eval_)

comparison = pd.DataFrame({
    "mean_predicted_cate": [s_cate.mean(), t_cate.mean(), x_cate.mean(), dr_cate.mean()],
    "std_predicted_cate": [s_cate.std(), t_cate.std(), x_cate.std(), dr_cate.std()],
}, index=["S-learner", "T-learner", "X-learner", "DR-learner"])

print(f"Actual ATE on eval set: {ate_eval:.5f}")
comparison

## Interpretation

DR-learner comes out with both the closest mean to the true ATE (0.00841 vs. 0.00951) and the lowest std (0.02153) of all four models. The propensity-weighted residual correction gives it a second, independent way to fix outcome-model error beyond what cross-imputation (X-learner) alone provides — it's not just "another smoothing step," it's grounded in the actual observed outcomes via the correction term, which pulls estimates back toward reality wherever the outcome models are imperfect.

Next: causal forest, then the decile/Qini evaluation to check which model's heterogeneity actually holds up on held-out data.

## Causal forest

Different in kind from the meta-learners above: instead of wrapping an outcome-prediction model, a causal forest is a tree-based model built to estimate treatment-effect heterogeneity *directly*. Ordinary tree splits minimize outcome prediction error; causal forest splits pick the feature cutoffs that maximize the *difference in treatment effect* between the resulting leaves.

**Honesty**: each tree uses one random subsample to decide where to split, and a separate, non-overlapping subsample to estimate the effect within each leaf. Without this, a leaf could get chosen because it *looked* like a big effect (partly by chance), and then that same chance-driven appearance would get "confirmed" by measuring on the same data — the same overfitting logic as evaluating a model on its own training set, just one level deeper (within a single tree).

Implemented via `econml.dml.CausalForestDML` (`model_y`/`model_t` play the same nuisance-model role as DR-learner's `mu`/`e` — this is the double-ML framework combined with honest-forest splitting for heterogeneity).

In [ ]:
from uplift.cate_models import fit_causal_forest, predict_causal_forest

cf_model = fit_causal_forest(train, outcome, n_estimators=100)
cf_cate = predict_causal_forest(cf_model, eval_)

comparison = pd.DataFrame({
    "mean_predicted_cate": [s_cate.mean(), t_cate.mean(), x_cate.mean(), dr_cate.mean(), cf_cate.mean()],
    "std_predicted_cate": [s_cate.std(), t_cate.std(), x_cate.std(), dr_cate.std(), cf_cate.std()],
}, index=["S-learner", "T-learner", "X-learner", "DR-learner", "Causal forest"])

print(f"Actual ATE on eval set: {ate_eval:.5f}")
comparison

## Decile evaluation (the real check)

Mean/std alone can't tell you whether a model's spread is *real* heterogeneity or noise — no individual ground-truth CATE ever exists to check against. The check that works: rank held-out users by predicted CATE, bin into deciles, and compute the **actual measured ATE** (real treatment/control comparison, not model predictions) within each bin. Real heterogeneity shows a genuine pattern; noise shows flat/random results regardless of how confident the model's predictions look.

In [ ]:
from uplift.evaluation import decile_table

models = {
    "S-learner": s_cate,
    "T-learner": t_cate,
    "X-learner": x_cate,
    "DR-learner": dr_cate,
    "Causal forest": cf_cate,
}

decile_tables_visit = {name: decile_table(eval_, cate, outcome) for name, cate in models.items()}

for name, tbl in decile_tables_visit.items():
    print(f"=== {name} ===")
    print(tbl[["decile", "n", "mean_predicted_cate", "actual_ate", "ci_low", "ci_high"]].to_string(index=False))
    print()

## Interpretation — `visit`

Consistent pattern across **all five models**: decile 9 (top 10% predicted uplift) shows a measured ATE roughly 5x the overall ATE (~0.046-0.052), with CIs clearly excluding zero. Deciles 0-8 are mostly flat and noisy, CIs crossing zero, no clear monotonic trend.

T-learner and causal forest showed the widest *predicted* spread (mean/std table above), but the decile table shows that extra spread doesn't translate into better discrimination across the range — it's concentrated in correctly flagging the same top decile the other models also find, not genuinely finer-grained ranking elsewhere. No model shows a clearly negative, CI-excluding-zero decile-0 result — no strong "sleeping dogs" evidence for `visit`.

**Practical takeaway**: all five models reliably find one thing — a real top ~10% high-uplift segment — and that's the trustworthy result to act on, not fine-grained ranking across the full population.

## Repeat for `conversion`

`conversion` is the outcome that actually matters commercially, but it's much rarer (0.29% base rate vs. `visit`'s 4.7%) — expect everything to be noisier. In particular: `model_control`'s training set has ~105K rows, but at a 0.2% conversion rate that's only ~210 positive-class examples for it to learn from — this compounds the "control group is smaller" problem from `visit` with "the thing being predicted barely ever happens within that already-small group."

In [ ]:
outcome = "conversion"
ate_eval = compute_ate(eval_, outcome).ate

s_model = fit_s_learner(train, outcome)
s_cate = predict_s_learner(s_model, eval_)

t_models = fit_t_learner(train, outcome)
t_cate = predict_t_learner(t_models, eval_)

x_model = fit_x_learner(train, outcome)
x_cate = predict_x_learner(x_model, eval_)

dr_model = fit_dr_learner(train, outcome)
dr_cate = predict_dr_learner(dr_model, eval_)

cf_model = fit_causal_forest(train, outcome, n_estimators=100)
cf_cate = predict_causal_forest(cf_model, eval_)

models = {
    "S-learner": s_cate,
    "T-learner": t_cate,
    "X-learner": x_cate,
    "DR-learner": dr_cate,
    "Causal forest": cf_cate,
}

comparison = pd.DataFrame({
    "mean_predicted_cate": [cate.mean() for cate in models.values()],
    "std_predicted_cate": [cate.std() for cate in models.values()],
}, index=list(models.keys()))

print(f"Actual ATE on eval set: {ate_eval:.5f}")
comparison

In [ ]:
decile_tables_conversion = {name: decile_table(eval_, cate, outcome) for name, cate in models.items()}

for name, tbl in decile_tables_conversion.items():
    print(f"=== {name} ===")
    print(tbl[["decile", "n", "mean_predicted_cate", "actual_ate", "ci_low", "ci_high"]].to_string(index=False))
    print()

## Interpretation — `conversion`

- **S-learner effectively collapsed**: predicted CATE is exactly `0.000000` for 9 of 10 deciles — not just shrunk, essentially non-functional for this rare outcome. Its apparent decile-9 result is likely an artifact of how ties get broken when ranking a mass of identical `0.0` predictions, not real discrimination.
- **Same top-decile pattern as `visit` reproduces**: decile 9 shows a real, CI-excluding-zero effect (~0.008-0.0098) roughly 5-7x the overall ATE (0.00146), consistent across T/X/DR-learner and causal forest.
- **Multiple comparisons caution**: with 10 deciles x 5 models = 50 CIs being scanned, ~2-3 would be expected to look "significant" by pure chance alone (5% false-positive rate per test) even under a true null. A couple of middle-decile results come out borderline — not reliable evidence of real heterogeneity there, just what pure noise predicts.

**Combined conclusion (`visit` + `conversion`)**: CATE modeling on this data reliably finds one thing — a real top ~10% high-uplift segment, consistent across both outcomes and all non-degenerate models. Beyond that slice, none of the models show trustworthy discrimination. This directly informs the policy curve step next: the practical targeting decision is likely closer to "target the top decile" than "finely rank the whole population."